# Porting of the Parseme 1.2 Shared Task in Codabench

This is the starting kit of the porting of the Parseme 1.2 shared task in Codabench. For more information about the shared task, visit this [link](https://multiword.sourceforge.net/PHITE.php?sitesig=CONF&page=CONF_02_MWE-LEX_2020___lb__COLING__rb__&subpage=CONF_40_Shared_Task). This starting kit aims to introduce new participants to the data format of the shared task, the expected predictions and how they are going to be scored.

In [1]:
import os
import sys
import numpy as np
import pandas as pd

scoring_path = os.path.abspath("programs")
if scoring_path not in sys.path:
    sys.path.append(scoring_path)

## Introduction

The datasets are available in the *input_data* folder. We will explore the format of the data given and to be submitted  by the participants through the example of the trial files.

In [2]:
data_dir = "trial"
!ls $data_dir*

EN-trial.raw.conllu	  EN-trial.test.cupt	   EN-trial.train.cupt
EN-trial.test.blind.cupt  EN-trial.test.pred.cupt


The files are in two formats:
- .conllu, the CoNLL-U format of Universal Dependencies. Tab-separated values (TSV) file composed of 10 columns (ID, FORM, LEMMA, UPOS, XPOS, FEATS, HEAD, DEPREL, DEPS, MISC). See more details: [CoNLL-U Format](https://universaldependencies.org/format.html)

In [3]:
with open(data_dir + "/EN-trial.raw.conllu", 'r') as f:
    lines = f.read().split('\n')
    for n in range(105,116): print(lines[n])

# text = Please double-check the email field and submit again.
1	Please	please	INTJ	UH	_	2	discourse	_	_
2	double	double	VERB	VB	Mood=Imp|VerbForm=Fin	0	root	_	SpaceAfter=No
3	-	-	PUNCT	,	_	2	punct	_	SpaceAfter=No
4	check	check	VERB	VB	Mood=Imp|VerbForm=Fin	2	parataxis	_	_
5	the	the	DET	DT	Definite=Def|PronType=Art	7	det	_	_
6	email	email	NOUN	NN	Number=Sing	7	compound	_	_
7	field	field	NOUN	NN	Number=Sing	4	obj	_	_
8	and	and	CCONJ	CC	_	9	cc	_	_
9	submit	submit	VERB	VB	Mood=Imp|VerbForm=Fin	2	conj	_	_
10	again	again	ADV	RB	_	9	advmod	_	SpaceAfter=No


- .cupt, the format of the Shared Task. It is an extension of the CoNLL-U format and stands for *CoNLL-U + parseme-TSV*. It is composed of 11 columns, the 10 from the CoNLL-U format and a new column PARSEME:MWE. It encodes information about verbal multiword expressions (VMWEs) present in a sentence. 

In [4]:
with open(data_dir + "/EN-trial.train.cupt", 'r') as f:
    lines = f.read().split('\n')
    for n in range(33, 44): print(lines[n])

# source_sent_id = http://hdl.handle.net/11234/1-2515 UD_English/en-ud-train.conllu newsgroup-groups.google.com_humanities.lit.authors.shakespeare_0018a7697318f71f_ENG_20031006_163200-0106
# text = But they determined to get rid forever of Garcia.
1	But	but	CCONJ	CC	_	3	cc	3:cc	_	*
2	they	they	PRON	PRP	Case=Nom|Number=Plur|Person=3|PronType=Prs	3	nsubj	3:nsubj	_	*
3	determined	determine	VERB	VBD	Mood=Ind|Tense=Past|VerbForm=Fin	0	root	0:root	_	*
4	to	to	PART	TO	_	5	mark	5:mark	_	*
5	get	get	VERB	VB	VerbForm=Inf	3	xcomp	3:xcomp	_	1:VID;2:IAV
6	rid	rid	ADJ	JJ	Degree=Pos	5	xcomp	5:xcomp	_	1;2
7	forever	forever	ADV	RB	_	5	advmod	5:advmod	_	*
8	of	of	ADP	IN	_	9	case	9:case	_	2
9	Garcia	Garcia	PROPN	NNP	Number=Sing	6	obl	6:obl	SpaceAfter=No	*


In the .cupt file, we can notice a new line starting with *#* before the sentence indicating the source of the image. We can also notice that the last column, PARSEME:MWE indicates either stars, numbers, or number and VMWE category labels.

The following extract tells us that there are two verbal multiword expressions in the previous sentence. A verbal idiom (VID) ***get rid*** and an inherently adpositional verbs (IAV) ***get rid of***. See more details: [Shared Task 1.2](https://gitlab.com/parseme/sharedtask-data/-/blob/master/1.2/README.md)

## Step 1: Exploratory Data Analysis

We can explore the train data using the following function to turn the .cupt file into a pandas DataFrame:

In [5]:
def extract_data(filename):
    with open(data_dir + '/' + filename, 'r') as f:
        table = []
        col = ["ID", "FORM", "LEMMA", "UPOS", "XPOS", "FEATS", "HEAD", "DEPREL", "DEPS", "MISC", "PARSEME:MWE"] 
        for line in f:
            if not line.startswith('\n') and not line.startswith('#'):
                table += [line.strip().split('\t')]
    df = pd.DataFrame(np.array(table), columns=col)
    return df         

df = extract_data("EN-trial.train.cupt")

Let's display the 16 first rows of the dataframe:

In [6]:
df.head(16)

,ID,FORM,LEMMA,UPOS,XPOS,FEATS,HEAD,DEPREL,DEPS,MISC,PARSEME:MWE
0,1,A,a,DET,DT,Definite=Ind|PronType=Art,3,det,3:det,_,1:VID
1,2,little,little,ADJ,JJ,Degree=Pos,3,amod,3:amod,_,1
2,3,birdie,birdie,NOUN,NN,Number=Sing,4,nsubj,4:nsubj,_,1
3,4,told,tell,VERB,VBD,Mood=Ind|Tense=Past|VerbForm=Fin,0,root,0:root,_,1
4,5,me,I,PRON,PRP,Case=Acc|Number=Sing|Person=1|PronType=Prs,4,iobj,4:iobj,_,*
5,6,that,that,SCONJ,IN,_,9,mark,9:mark,_,*
6,7,you,you,PRON,PRP,Case=Nom|Person=2|PronType=Prs,9,nsubj,9:nsubj,_,*
7,8,were,be,AUX,VBD,Mood=Ind|Tense=Past|VerbForm=Fin,9,aux,9:aux,_,*
8,9,checking,check,VERB,VBG,Tense=Pres|VerbForm=Part,4,ccomp,4:ccomp,_,2:VPC.semi
9,10,into,into,ADP,IN,_,11,case,11:case,_,2


We can see more clearly the format of the datasets. New sentences start when the ID goes back to 1.

In [7]:
df.describe()

,ID,FORM,LEMMA,UPOS,XPOS,FEATS,HEAD,DEPREL,DEPS,MISC,PARSEME:MWE
count,109,109,109,109,109,109,109,109,109,109,109
unique,27,83,76,13,24,27,21,24,87,2,16
top,1,.,.,VERB,IN,_,3,nsubj,0:root,_,*
freq,7,6,6,18,14,38,17,11,7,98,80


## Step 2: Making predictions

todo

## Step 3: Scoring results

Once the model produced a test.system.cupt file from the test.blind.cupt file, the scoring program will be able to compare it to test.cupt. To do so, the tsvlib is used:

In [8]:
import tsvlib

Its iter_tsv_sentence(filename) function will yield the sentences in a TSVSentence class containing TSVToken for every word in the original sentence. See the following example:

In [9]:
sentences = tsvlib.iter_tsv_sentences(open(data_dir + "/EN-trial.test.cupt", 'r'))
print(sentences)
print()
print(str(next(sentences))[:656])

<generator object iter_tsv_sentences at 0x7fc0b0f19540>

TSVSentence('trial/EN-trial.test.cupt', 4, [{'ID': '1', 'FORM': 'My', 'LEMMA': 'my', 'UPOS': 'PRON', 'XPOS': 'PRP$', 'FEATS': 'Number=Sing|Person=1|Poss=Yes|PronType=Prs', 'HEAD': '2', 'DEPREL': 'nmod:poss', 'DEPS': '2:nmod:poss', 'PARSEME:MWE': '*'}, {'ID': '2', 'FORM': 'suggestion', 'LEMMA': 'suggestion', 'UPOS': 'NOUN', 'XPOS': 'NN', 'FEATS': 'Number=Sing', 'HEAD': '3', 'DEPREL': 'nsubj', 'DEPS': '3:nsubj', 'PARSEME:MWE': '*'}, {'ID': '3', 'FORM': 'is', 'LEMMA': 'be', 'UPOS': 'VERB', 'XPOS': 'VBZ', 'FEATS': 'Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin', 'HEAD': '0', 'DEPREL': 'root', 'DEPS': '0:root', 'PARSEME:MWE': '*'}, {'ID': '4', '


The evaluate.py script will use this tsvlib to go through the data and count the different multiword expression and information to produce scores and statistics:

In [10]:
%run ../scoring_program/evaluate.py -h

usage: evaluate.py [-h] [--debug] [--combinatorial] [--train train_file]
                   [--dev dev_file] --gold gold_file --pred prediction_file

Evaluate input `prediction` against `gold`.

options:
  -h, --help            show this help message and exit
  --debug               Print extra debugging information (you can grep it,
                        and should probably pipe it into `less -SR`)
  --combinatorial       Run O(n!) algorithm for weighted bipartite matching.
                        You should probably not use this option
  --train train_file    The training file in .cupt format (to calculate
                        statistics regarding seen MWEs)
  --dev dev_file        The dev file in .cupt format (to calculate statistics
                        regarding seen MWEs)
  --gold gold_file      The reference gold-standard file in .cupt format
  --pred prediction_file
                        The system prediction file in .cupt format


The evaluate.py script is used with subprocess to produce and collect the scores:

>ℹ️ **NOTE**
>
>The trial data notation is different. Here the predicted file is called EN-trial.test.pred.cupt but for your predictions, the files must be called test.system.cupt

In [11]:
if os.path.exists(os.path.abspath('scoring_program/evaluate.py')):
    print('yes')
else:
    print(os.path.abspath('scoring_program/evaluate.py'))

/home/achille/portage-codabench-parseme-1.2/bundle_parseme/starting_kit/scoring_program/evaluate.py


In [12]:
import evaluate
import subprocess
import re

result = subprocess.run(['python3', 'programs/evaluate.py', '--train', 'trial/EN-trial.train.cupt', '--pred', 'trial/EN-trial.test.pred.cupt', '--gold', 'trial/EN-trial.test.cupt'], capture_output=True, text=True)
content = result.stdout.split('\n')
for line in content:
    print(line)

## Global evaluation
* MWE-based: P=6/13=0.4615 R=6/10=0.6000 F=0.5217
* Tok-based: P=13/28=0.4643 R=13/20=0.6500 F=0.5417

## Per-category evaluation (partition of Global)
* IAV: MWE-proportion: gold=1/10=10% pred=2/13=15%
* IAV: MWE-based: P=0/2=0.0000 R=0/1=0.0000 F=0.0000
* IAV: Tok-based: P=2/5=0.4000 R=2/3=0.6667 F=0.5000
* LVC.cause: MWE-proportion: gold=1/10=10% pred=1/13=8%
* LVC.cause: MWE-based: P=0/1=0.0000 R=0/1=0.0000 F=0.0000
* LVC.cause: Tok-based: P=1/2=0.5000 R=1/2=0.5000 F=0.5000
* LVC.full: MWE-proportion: gold=3/10=30% pred=7/13=54%
* LVC.full: MWE-based: P=1/7=0.1429 R=1/3=0.3333 F=0.2000
* LVC.full: Tok-based: P=4/16=0.2500 R=4/6=0.6667 F=0.3636
* VID: MWE-proportion: gold=3/10=30% pred=1/13=8%
* VID: MWE-based: P=1/1=1.0000 R=1/3=0.3333 F=0.5000
* VID: Tok-based: P=1/1=1.0000 R=1/5=0.2000 F=0.3333
* VPC.full: MWE-proportion: gold=2/10=20% pred=2/13=15%
* VPC.full: MWE-based: P=1/2=0.5000 R=1/2=0.5000 F=0.5000
* VPC.full: Tok-based: P=2/4=0.5000 R=2/4=0.5000 F=0.

In the previous code snippet, we have a list of strings, it is then parsed to produce the dictionnary that will later be turn into a json for the learderboard:

In [13]:
sub_dict = {}
current_category = ""
for line in content:
    if line.startswith('#'):
        p = re.compile(r"## (?P<category>[a-zA-Z\- ]+)[a-zA-Z\(\) ]*")
        _match = p.match(line)
        match _match.group("category"):
            case "Global evaluation":
                current_category = "Global"
            case "Per-category evaluation ":
                current_category = "Per-category"
            case "MWE continuity ":
                current_category = "Continuity"
            case "Number of tokens ":
                current_category = "Number-of-tokens"
            case "Whether seen in train ":
                current_category = "Un-seen"
            case "Whether identical to train ":
                current_category = "Variation"
            case _:
                print("ERROR: category not recognized")
                sys.exit(1)
        sub_dict[current_category] = {}
    elif line and current_category:
        if ("MWE-based" in line) or ("Tok-based" in line):
            p = re.compile(r"\* (?P<type>[a-zA-Z\-\.]+): ((?P<subtype>[a-zA-Z\-]+): )?P=[0-9\/\=]+(?P<precision>[0-9\.]+) R=[0-9\/\=]+(?P<recall>[0-9\.]+) F=(?P<f_score>[0-9\.]+)")
            _match = p.match(line)
            if current_category == "Global":
                sub_dict[current_category][_match.group("type")] = [float(_match.group("precision")), float(_match.group("recall")), float(_match.group("f_score"))]
            elif current_category == "Per-category":
                if _match.group("type") not in sub_dict[current_category]:
                    sub_dict[current_category][_match.group("type")] = {}
                sub_dict[current_category][_match.group("type")][_match.group("subtype")] = [float(_match.group("precision")), float(_match.group("recall")), float(_match.group("f_score"))]
            else:
                sub_dict[current_category][_match.group("type") + ' ' + _match.group("subtype")] = [float(_match.group("precision")), float(_match.group("recall")), float(_match.group("f_score"))]
print(sub_dict)

{'Global': {'MWE-based': [0.4615, 0.6, 0.5217], 'Tok-based': [0.4643, 0.65, 0.5417]}, 'Per-category': {'IAV': {'MWE-based': [0.0, 0.0, 0.0], 'Tok-based': [0.4, 0.6667, 0.5]}, 'LVC.cause': {'MWE-based': [0.0, 0.0, 0.0], 'Tok-based': [0.5, 0.5, 0.5]}, 'LVC.full': {'MWE-based': [0.1429, 0.3333, 0.2], 'Tok-based': [0.25, 0.6667, 0.3636]}, 'VID': {'MWE-based': [0.0, 0.3333, 0.5], 'Tok-based': [0.0, 0.2, 0.3333]}, 'VPC.full': {'MWE-based': [0.5, 0.5, 0.5], 'Tok-based': [0.5, 0.5, 0.5]}}, 'Continuity': {'Continuous MWE-based': [0.5, 0.5, 0.5], 'Discontinuous MWE-based': [0.4444, 0.6667, 0.5333]}, 'Number-of-tokens': {'Multi-token MWE-based': [0.4167, 0.5556, 0.4762], 'Single-token MWE-based': [0.0, 0.0, 1.0]}, 'Un-seen': {'Seen-in-train MWE-based': [0.5, 0.5, 0.5], 'Unseen-in-train MWE-based': [0.4286, 0.75, 0.5455]}, 'Variation': {'Variant-of-train MWE-based': [0.3333, 0.25, 0.2857], 'Identical-to-train MWE-based': [0.6667, 0.0, 0.8]}}


In [14]:
general_unseen = sub_dict["Un-seen"]["Unseen-in-train MWE-based"]
general_mwe = sub_dict["Global"]["MWE-based"]
general_tok = sub_dict["Global"]["Tok-based"]
labels = [x + y for x in ["Unseen MWE-based ", "Global MWE-based ", "Global Token-based "] for y in ['P', 'R', 'F']]
general = {label: value for label, value in zip(labels, general_unseen + general_mwe + general_tok)}

In [15]:
print(general)
print(general_unseen)

{'Unseen MWE-based P': 0.4286, 'Unseen MWE-based R': 0.75, 'Unseen MWE-based F': 0.5455, 'Global MWE-based P': 0.4615, 'Global MWE-based R': 0.6, 'Global MWE-based F': 0.5217, 'Global Token-based P': 0.4643, 'Global Token-based R': 0.65, 'Global Token-based F': 0.5417}
[0.4286, 0.75, 0.5455]
